# Jhadina — encrypted Restic Drive drill (synthetic only)

**Goal:** With your own Google Colab session and the private `JHADINA-HOMEBASE/01-BACKUPS` folder, back up **64 random bytes** using Restic encryption, download them by exact immutable snapshot ID, and verify their SHA-256. This uses the *mounted My Drive directory*, not machine rclone OAuth.

**Limitations:** This is NOT a backup of Supabase, PostgreSQL, MinIO, NATS, personal files, or a server. It is temporary Colab compute. The disposable encryption passphrase exists only in the running session; the demonstration backup is not recoverable after that session ends unless the key is separately retained. Do not put any real files or recovery keys in this test.

Review every cell, authorize Google Drive access only if comfortable, then run in order. Neither cell deletes anything from Google Drive.


In [ ]:
from google.colab import drive
drive.mount('/content/drive')  # authorize with the owner Google account


In [ ]:
import hashlib, json, os, re, secrets, shutil, subprocess, tempfile
from pathlib import Path

root = Path('/content/drive/MyDrive/JHADINA-HOMEBASE/01-BACKUPS')
if not root.is_dir() or root.is_symlink():
    raise RuntimeError('Expected private JHADINA-HOMEBASE/01-BACKUPS folder is not mounted')
parent = root / '_JHADINA_COLAB_SYNTHETIC_RESTIC'
if parent.is_symlink() or (parent.exists() and not parent.is_dir()):
    raise RuntimeError('Unexpected synthetic test parent: stop without writing')
run_id = secrets.token_hex(12)
trial_root = parent / run_id
if trial_root.exists() or trial_root.is_symlink():
    raise RuntimeError('Unique synthetic test folder already exists; stop')
trial_root.mkdir(parents=True, mode=0o700)
if not shutil.which('restic'):
    subprocess.run(['apt-get', 'update', '-qq'], check=True, stdout=subprocess.DEVNULL, timeout=300)
    subprocess.run(['apt-get', 'install', '-y', '-qq', 'restic'], check=True, stdout=subprocess.DEVNULL, timeout=300)

with tempfile.TemporaryDirectory(prefix='jhadina-restic-synthetic-',dir='/content') as scratch:
    work = Path(scratch)
    os.chmod(work, 0o700)
    secret_file = work / 'password'
    fd = os.open(secret_file, os.O_WRONLY | os.O_CREAT | os.O_EXCL, 0o600)
    with os.fdopen(fd, 'w') as handle:
        handle.write(secrets.token_urlsafe(48))
    env = {**os.environ, 'RESTIC_PASSWORD_FILE': str(secret_file)}
    repo = trial_root / 'repository'
    def restic(*args, input_bytes=None, timeout=300):
        return subprocess.run(['restic','-r',str(repo),*args], input=input_bytes,
                              stdout=subprocess.PIPE,stderr=subprocess.PIPE,
                              env=env,check=True,timeout=timeout)
    restic('init', timeout=180)
    original = secrets.token_bytes(64)  # ONLY synthetic data enters Google Drive
    sha = hashlib.sha256(original).hexdigest()
    written = restic('backup','--stdin','--stdin-filename','synthetic.bin','--json',
                     input_bytes=original,timeout=600)
    summaries = [json.loads(line) for line in written.stdout.splitlines()
                 if line.strip() and json.loads(line).get('message_type') == 'summary']
    if len(summaries) != 1:
        raise RuntimeError('Restic did not return exactly one immutable snapshot')
    snapshot = summaries[0].get('snapshot_id','')
    if not re.fullmatch(r'[0-9a-f]{8,64}',snapshot):
        raise RuntimeError('Invalid immutable Restic snapshot ID')
    restored = restic('dump',snapshot,'/synthetic.bin',timeout=600).stdout
    if len(restored) != 64 or hashlib.sha256(restored).hexdigest() != sha:
        raise RuntimeError('ENCRYPTED RESTORE FAILED: unexpected size/hash')
    print(json.dumps({
        'schema':'jhadina.google-homebase.colab-restic-synthetic.v1',
        'synthetic_bytes':64, 'sha256':sha, 'restic_snapshot_id':snapshot,
        'encrypted_mounted_drive_roundtrip_verified':True,
        'machine_rclone_oauth_verified':False,
        'server_unattended_backup_verified':False,
        'postgres_database_restore_verified':False,
        'minio_nats_recovery_verified':False,
        'encryption_password_recovery_after_session_verified':False,
        'production_backup_restored':False
    },indent=2))


**After the result:** Do not share or save the Colab session's temporary Restic password. This repository holds only throwaway synthetic data. The printed SHA and snapshot ID are safe test evidence but **do not prove you can recover this temporary repository after Colab disconnects**. Real recovery requires owner-controlled credentials and tested key escrow.
